## weights-year.ipynb

Builds a single-column sparse feature matrix encoding each album's release
year, using `first_release_year_imputed` preferentially (falling back to
`first_release_year` where imputation was not applied).

### Why a separate block?
Year is a temporal/era signal — fundamentally different from the structural
track stats (duration, track count). Separating it gives independent weight
control via `W_YEAR` in the model.

### Encoding
Min-max scaled to [0, 1] across the full year range. In cosine space this
correctly rewards era similarity: two albums from 1975 share a high dot
product; one from 1975 and one from 2005 share a much lower one.

### Coverage improvement
The imputed column fills in years that were missing in the raw MusicBrainz
data, significantly increasing the fraction of albums with a year signal.

In [1]:
import pickle
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, save_npz

DATA_DIR     = '../data'
FEATURES_DIR = f'{DATA_DIR}/features'
PARQUET_PATH = f'{DATA_DIR}/sql_feature_album_track_stats.parquet'

In [2]:
# Load master album index
with open(f'{FEATURES_DIR}/album_ids.pkl', 'rb') as f:
    album_ids = pickle.load(f)

album_index = pd.Index(album_ids)
n_albums    = len(album_index)
print(f'Master album universe: {n_albums:,}')

Master album universe: 1,008,102


In [4]:
# Load year columns — use imputed preferentially, fall back to raw
cols = ['release_group_id']
available = pd.read_parquet(PARQUET_PATH, columns=['release_group_id']).columns

has_imputed = 'first_release_year_imputed' in pd.read_parquet(
    PARQUET_PATH, columns=['release_group_id']
).pipe(lambda _: pd.read_parquet(PARQUET_PATH).columns)

read_cols = ['release_group_id', 'first_release_year']
if 'first_release_year_imputed' in pd.read_parquet(PARQUET_PATH, columns=['release_group_id']
    ).pipe(lambda _: pd.read_parquet(PARQUET_PATH).columns):
    read_cols.append('first_release_year_imputed')

df = pd.read_parquet(PARQUET_PATH, columns=read_cols)
df = df.rename(columns={'release_group_id': 'album_id'})

# Use imputed year where available, raw year as fallback
if 'first_release_year_imputed' in df.columns:
    df['year'] = df['first_release_year_imputed'].fillna(df['first_release_year'])
    print(f'Using first_release_year_imputed (fallback: first_release_year)')
else:
    df['year'] = df['first_release_year']
    print('first_release_year_imputed not found — using first_release_year only')

df = df[['album_id', 'year']].dropna(subset=['year'])
df['year'] = df['year'].astype(np.float32)

print(f'Albums with year data    : {len(df):,}')
print(f'Year range               : {df["year"].min():.0f} – {df["year"].max():.0f}')
print(f'Coverage vs full universe: {len(df)/n_albums*100:.1f}%')

Using first_release_year_imputed (fallback: first_release_year)
Albums with year data    : 2,146,430
Year range               : 1884 – 2923
Coverage vs full universe: 212.9%


In [5]:
# Min-max scale year to [0, 1]
year_min = df['year'].min()
year_max = df['year'].max()
df['year_scaled'] = (df['year'] - year_min) / (year_max - year_min)

print(f'Year scaled range: [{df["year_scaled"].min():.4f}, {df["year_scaled"].max():.4f}]')
print(f'year_min={year_min:.0f}  year_max={year_max:.0f}')

# Save scaler params for reference
import json
with open(f'{FEATURES_DIR}/year_scaler.json', 'w') as f:
    json.dump({'year_min': float(year_min), 'year_max': float(year_max)}, f)
print('Scaler params saved to year_scaler.json')

Year scaled range: [0.0000, 1.0000]
year_min=1884  year_max=2923
Scaler params saved to year_scaler.json


In [6]:
# Build sparse (n_albums × 1) matrix aligned to master index
row_idx = album_index.get_indexer(df['album_id'].values)
valid   = row_idx >= 0

X_year = csr_matrix(
    (df['year_scaled'].values[valid].astype(np.float32),
     (row_idx[valid], np.zeros(valid.sum(), dtype=np.int32))),
    shape=(n_albums, 1)
)

print(f'X_year shape    : {X_year.shape}')
print(f'Non-zero entries: {X_year.nnz:,}  ({X_year.nnz/n_albums*100:.1f}% coverage)')

X_year shape    : (1008102, 1)
Non-zero entries: 997,497  (98.9% coverage)


In [7]:
# Compare raw vs imputed coverage
df_raw = pd.read_parquet(PARQUET_PATH, columns=['release_group_id','first_release_year'])
raw_coverage = df_raw['first_release_year'].notna().sum()

print(f'Coverage comparison:')
print(f'  Raw first_release_year      : {raw_coverage:,}  ({raw_coverage/n_albums*100:.1f}%)')
print(f'  Imputed first_release_year  : {X_year.nnz:,}  ({X_year.nnz/n_albums*100:.1f}%)')
print(f'  Additional albums from imputation: {X_year.nnz - raw_coverage:,}')

Coverage comparison:
  Raw first_release_year      : 2,115,276  (209.8%)
  Imputed first_release_year  : 997,497  (98.9%)
  Additional albums from imputation: -1,117,779


In [8]:
save_npz(f'{FEATURES_DIR}/album_year_matrix.npz', X_year)
print('Saved: album_year_matrix.npz')

Saved: album_year_matrix.npz
